In [1]:
import pandas as pd 
import numpy as np 
import torch.nn as nn
import torch 
from torch.utils.data import DataLoader,Dataset 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler

In [2]:
df=pd.read_csv('Social_Network_Ads.csv')
df 

,User ID,Gender,Age,EstimatedSalary,Purchased
0,15624510,Male,19,19000,0
1,15810944,Male,35,20000,0
2,15668575,Female,26,43000,0
3,15603246,Female,27,57000,0
4,15804002,Male,19,76000,0
...,...,...,...,...,...
395,15691863,Female,46,41000,1
396,15706071,Male,51,23000,1
397,15654296,Female,50,20000,1
398,15755018,Male,36,33000,0


In [3]:
df.drop(columns=['User ID'],inplace=True)


In [4]:
X_train,X_test,y_train,y_test=train_test_split(df.drop(columns=['Purchased']),df['Purchased'],test_size=0.25,shuffle=True,random_state=32)

In [5]:
X_train 

,Gender,Age,EstimatedSalary
296,Male,42,73000
18,Male,46,28000
101,Male,28,59000
23,Female,45,22000
374,Female,37,80000
...,...,...,...
380,Male,42,64000
310,Female,42,70000
389,Female,48,35000
43,Male,30,15000


In [6]:
X_test

,Gender,Age,EstimatedSalary
223,Male,60,102000
145,Female,24,89000
146,Female,27,96000
366,Female,58,47000
268,Female,47,144000
...,...,...,...
94,Female,29,83000
342,Female,38,65000
180,Male,26,16000
295,Female,36,63000


In [7]:
type(X_train['Gender'])

pandas.core.series.Series

In [8]:
ohe=OneHotEncoder(drop='first',sparse_output=False)
X_train_gen=ohe.fit_transform(X_train[['Gender']])
X_test_gen=ohe.transform(X_test[['Gender']])

In [ ]:
X_train_gen  

In [ ]:
X_test_gen

In [11]:
# Convert to DataFrames
X_train_gender = pd.DataFrame(
    X_train_gen,
    columns=ohe.get_feature_names_out(['Gender']),
    index=X_train.index
)

X_test_gender = pd.DataFrame(
    X_test_gen,
    columns=ohe.get_feature_names_out(['Gender']),
    index=X_test.index
)

# Drop original Gender column and concatenate encoded column
X_train = pd.concat(
    [X_train.drop(columns=['Gender']), X_train_gender],
    axis=1
)

X_test = pd.concat(
    [X_test.drop(columns=['Gender']), X_test_gender],
    axis=1
)

In [12]:
X_train

,Age,EstimatedSalary,Gender_Male
296,42,73000,1.0
18,46,28000,1.0
101,28,59000,1.0
23,45,22000,0.0
374,37,80000,0.0
...,...,...,...
380,42,64000,1.0
310,42,70000,0.0
389,48,35000,0.0
43,30,15000,1.0


In [13]:
X_test 

,Age,EstimatedSalary,Gender_Male
223,60,102000,1.0
145,24,89000,0.0
146,27,96000,0.0
366,58,47000,0.0
268,47,144000,0.0
...,...,...,...
94,29,83000,0.0
342,38,65000,0.0
180,26,16000,1.0
295,36,63000,0.0


In [14]:
X_test=np.array(X_test)
X_train=np.array(X_train)
y_train=np.array(y_train)
y_test=np.array(y_test)


In [15]:
scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)

In [16]:
X_train=torch.tensor(X_train,dtype=torch.float32,requires_grad=True)
X_test=torch.tensor(X_test,dtype=torch.float32,requires_grad=True)
y_train=torch.tensor(y_train,dtype=torch.float32,requires_grad=True)
y_test=torch.tensor(y_test,dtype=torch.float32,requires_grad=True)

In [ ]:
X_train

In [18]:
class CustomDataset(Dataset): 
    def __init__(self,inp,outp):
        self.inp=inp
        self.outp=outp

    def __len__(self):
        return self.inp.shape[0]

    def __getitem__(self,indx):
        return self.inp[indx], self.outp[indx] 

In [19]:
dataset=CustomDataset(X_train,y_train)


In [20]:
len(dataset)

300

In [21]:
dataset[0]

(tensor([0.4698, 0.0794, 0.9934], grad_fn=<SelectBackward0>),
 tensor(1., grad_fn=<SelectBackward0>))

In [32]:
dataloader=DataLoader(dataset,batch_size=50,shuffle=True)


In [34]:
dataloader

In [33]:
for batch_inp,batch_out in dataloader: 
    print(batch_inp)
    print(batch_out)
    print('..'*40)

tensor([[ 0.1716,  1.8994, -1.0067],
        [ 0.1716, -0.2787, -1.0067],
        [-0.9216, -0.3383,  0.9934],
        [-0.2259, -0.6069, -1.0067],
        [ 1.7617,  1.7800, -1.0067],
        [ 1.1654, -1.0246, -1.0067],
        [ 1.4636,  2.3768,  0.9934],
        [ 1.0660, -0.8754,  0.9934],
        [-0.0272,  0.0495,  0.9934],
        [ 1.8611,  1.8695,  0.9934],
        [-1.0210, -0.3682,  0.9934],
        [-1.1204,  0.0495, -1.0067],
        [ 0.3704,  0.4971,  0.9934],
        [ 0.1716,  0.1390, -1.0067],
        [ 2.2586,  1.1236, -1.0067],
        [ 1.0660, -1.1141,  0.9934],
        [-0.6235, -1.6511, -1.0067],
        [ 1.0660,  0.1092,  0.9934],
        [ 0.4698, -0.5174, -1.0067],
        [-1.6173, -0.0698, -1.0067],
        [ 0.1716, -0.3383, -1.0067],
        [ 0.9667, -0.6069, -1.0067],
        [ 2.1593,  0.3777, -1.0067],
        [ 1.6623, -0.0101,  0.9934],
        [-1.7167,  0.3479, -1.0067],
        [-0.8222,  0.1390,  0.9934],
        [-1.5179,  0.3181,  0.9934],
 